# 10_01 Attention by hand: three words, three matrices, one softmax

Attention is a handful of matrix products and one softmax. In this notebook you compute it for the chapter's
phrase "Action gets results" in plain NumPy, one step at a time, and then watch PyTorch's own attention
function produce exactly the same numbers. Then you add the mask a decoder uses so it cannot read ahead, and
take a real `nn.MultiheadAttention` layer apart to see that it does nothing else.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-10-attention-is-the-whole-trick", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import math
import os
import numpy as np
import torch
import torch.nn.functional as F
from nlpcheck import ask, guess, reveal, check_10_01, action_gets_results

np.set_printoptions(precision=3, suppress=True)
print("PyTorch", torch.__version__, "| NumPy", np.__version__)

## 1. Recall

**r1.** In the encoder-decoder of the last lab, what did the decoder receive from the encoder?
(a) every hidden state the encoder produced, (b) the encoder's last hidden state only, one vector,
(c) the input words themselves

In [ ]:
ask("r1", "")

**r2.** Softmax turns the scores `[2.0, 1.0, 0.0]` into weights. Which is true of the result?
(a) they are all positive and add up to 1, (b) they are 2/3, 1/3 and 0, (c) the largest is exactly twice the middle one

In [ ]:
ask("r2", "")

## 2. The worked example: self-attention for "Action gets results"

Each of the three words arrives as an embedding of four numbers, one row of `X`. The layer has learned three
matrices, `W_q`, `W_k` and `W_v`. Here they are made from a fixed seed so everyone sees the same numbers; in
a trained model they are what training found. The recipe, from the chapter:

1. `Q = X @ W_q`, `K = X @ W_k`, `V = X @ W_v`: every word gets a query, a key and a value.
2. `scores = Q @ K.T`: row *i* holds word *i*'s query dotted with every word's key.
3. Divide by `sqrt(d_k)`, the square root of the key length.
4. Softmax along each row, so each word's weights are positive and add up to 1.
5. `Z = weights @ V`: each word's output is a weighted average of all the values.

In [ ]:
words, X, W_q, W_k, W_v = action_gets_results()
print("X, one row per word:"); print(X)

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))   # subtracting the max changes nothing but avoids overflow
    return e / e.sum(axis=-1, keepdims=True)

Q, K, V = X @ W_q, X @ W_k, X @ W_v          # (3, 4) each
d_k = K.shape[-1]
scores = Q @ K.T / math.sqrt(d_k)             # (3, 3): every query against every key
A = softmax(scores)                           # (3, 3): each row adds up to 1
Z = A @ V                                     # (3, 4): one output per word
for w, row in zip(words, A):
    print(f"{w:>8} attends:", dict(zip(words, row.round(3))))

Read one row. "Action" spreads its attention over all three words, itself included, and its output `Z[0]` is
that mixture of the three value vectors. Nothing here was trained to do anything yet, so the weights are not
meaningful; the arithmetic is the point.

Now the same thing from PyTorch. `F.scaled_dot_product_attention` is the function every transformer in
PyTorch calls, on a laptop or on a data-centre GPU. It takes the queries, keys and values and does steps 2
to 5 in one call.

In [ ]:
Z_torch = F.scaled_dot_product_attention(torch.tensor(Q), torch.tensor(K), torch.tensor(V))
print("largest difference from your NumPy Z:", np.abs(Z_torch.numpy() - Z).max())

A difference of zero, or a few units in the fifteenth decimal place: the last digits of floating-point
arithmetic. There is no hidden ingredient.

## 3. Your turn: the causal mask

A decoder writes one word at a time, so when it computes word 2 it must not look at word 3, which does not
exist yet. The fix is a **mask**: before the softmax, every score above the diagonal is set to minus infinity,
and `exp(-inf)` is exactly 0, so those positions get no weight at all.

Fill in the two lines. `np.triu(np.ones((3, 3), dtype=bool), k=1)` is `True` above the diagonal, and
`np.where(mask, -np.inf, scores)` swaps those scores for minus infinity.

In [ ]:
mask = None             # YOUR CODE HERE: True above the diagonal, False on and below it
A_causal = None         # YOUR CODE HERE: softmax of the scores with the masked ones set to -inf
Z_causal = A_causal @ V if A_causal is not None else None

if A_causal is not None:
    print(A_causal)
    Zt = F.scaled_dot_product_attention(torch.tensor(Q), torch.tensor(K), torch.tensor(V), is_causal=True)
    print("largest difference from PyTorch's is_causal=True:", np.abs(Zt.numpy() - Z_causal).max())

The first row is `[1, 0, 0]`: the first word can only attend to itself. The last row is the same as before,
because the last word was always allowed to see everything.

## 4. Take a real layer apart

`nn.MultiheadAttention` is the layer the transformer block in the next notebook uses. It runs attention
several times in parallel, each **head** with its own slice of the query, key and value projections, then
joins the heads and mixes them with one more matrix.

Before you run the next cell: a layer for 8-dimensional inputs, with 1 head, then 2 heads, then 4 heads.
More heads means more attention computations. How many weights does each have?
(a) 2 heads twice as many as 1, (b) all the same, (c) more heads, fewer weights

In [ ]:
guess("heads_weights", None)   # "a", "b" or "c" 

In [ ]:
counts = {h: sum(p.numel() for p in torch.nn.MultiheadAttention(8, h).parameters()) for h in (1, 2, 4)}
print("weights by number of heads:", counts)
reveal("heads_weights", "b")

All the same, 288. The heads do not add weights, they **split** them: with 8 dimensions and 4 heads, each
head's queries, keys and values are 2 numbers long instead of 8. Each head gets a narrower view, and the
point of having several is that each can attend to something different.

Now the proof that the layer is nothing but the arithmetic above. The cell pulls the layer's weights out, runs
two heads by hand in NumPy, joins them and applies the output matrix.

In [ ]:
torch.manual_seed(0)
mha = torch.nn.MultiheadAttention(4, 2, bias=False, batch_first=True)
Wq, Wk, Wv = [w.detach().numpy() for w in mha.in_proj_weight.chunk(3)]   # each (4, 4); PyTorch stores W.T
Wo = mha.out_proj.weight.detach().numpy()

q, k, v = X @ Wq.T, X @ Wk.T, X @ Wv.T
heads = []
for h in range(2):                                   # head h owns columns 2h and 2h + 1
    s = slice(2 * h, 2 * h + 2)
    heads.append(softmax(q[:, s] @ k[:, s].T / math.sqrt(2)) @ v[:, s])
by_hand = np.concatenate(heads, axis=1) @ Wo.T

xt = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
layer_out, _ = mha(xt, xt, xt)
mha_max_diff = float(np.abs(layer_out[0].detach().numpy() - by_hand).max())
print("largest difference between the layer and your hand-run heads:", mha_max_diff)

In [ ]:
os.makedirs("out", exist_ok=True)
tolist = lambda a: a.tolist() if a is not None else None
json.dump({"A": A.tolist(), "Z": Z.tolist(), "A_causal": tolist(A_causal), "Z_causal": tolist(Z_causal),
           "mha_max_diff": mha_max_diff}, open("out/10_01_attention.json", "w"), indent=1)
check_10_01()

## 5. Exit ticket

**x1.** The book's alignment scores for "The", "FBI" and "is", seen from "chasing", come out as
`[0.2, 0.5, 0.3]`. What is the context vector for "chasing"?
(a) the value vector of "FBI", the largest weight, (b) the average of the three value vectors,
(c) 0.2, 0.5 and 0.3 times the three value vectors, added up

In [ ]:
ask("x1", "")

Explain it back: why does the causal mask set scores to minus infinity rather than setting the weights to zero
after the softmax? One or two sentences.

*Your explanation:* 